# AI/ML for Retail Inventory Management: Demand Forecasting

## Project Overview

**Author:** AI/ML Learning Series  
**Date:** April 2026  
**Use Case:** Retail Inventory Optimization through Demand Forecasting

---

### 1. Introduction (10-15 words)
**Machine learning models predict product demand to optimize retail inventory levels efficiently.**

---

### 2. Description (10-15 words)
**Time series forecasting using historical sales data to predict future product demand.**

---

### 3. Objective (10-15 words)
**Minimize stockouts and overstock by accurately forecasting demand using machine learning techniques.**

---

### 4. Process (10-15 words)
**Data collection, exploration, feature engineering, model training, evaluation, and deployment preparation.**

---

### 5. Tools and Technologies Used (10-15 words)
**Python, Pandas, Scikit-learn, XGBoost, Prophet, Matplotlib, Seaborn for data analysis and modeling.**

---

### 6. Value Proposition (10-15 words)
**Reduce inventory costs, improve customer satisfaction, and increase profitability through predictive analytics.**

---

### 7. References (10-15 words)
**Scikit-learn documentation, Facebook Prophet, retail analytics research papers, and industry case studies.**

---

## Table of Contents

1. [Business Problem](#business-problem)
2. [Data Generation & Exploration](#data-generation)
3. [Feature Engineering](#feature-engineering)
4. [Model Building](#model-building)
5. [Model Evaluation](#model-evaluation)
6. [Business Insights & Recommendations](#insights)
7. [Deployment Considerations](#deployment)

## 1. Business Problem <a name="business-problem"></a>

### The Challenge

Retail businesses face a critical inventory management challenge:

- **Overstocking** leads to:
  - High holding costs
  - Product obsolescence
  - Tied-up capital
  
- **Understocking** results in:
  - Lost sales opportunities
  - Customer dissatisfaction
  - Damage to brand reputation

### The Solution

**Demand Forecasting using Machine Learning** helps predict future product demand based on:
- Historical sales patterns
- Seasonal trends
- Promotional activities
- External factors (holidays, weather, etc.)

This enables:
- Optimized inventory levels
- Reduced costs
- Improved customer satisfaction

## 2. Import Libraries and Setup <a name="setup"></a>

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Advanced models
import xgboost as xgb

# Warning suppression for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 3. Data Generation & Exploration <a name="data-generation"></a>

For this learning project, we'll create synthetic retail sales data that mimics real-world patterns:
- **Seasonality**: Higher sales during certain months
- **Trend**: Overall growth or decline over time
- **Promotions**: Sales spikes during promotional periods
- **Day-of-week effects**: Different patterns on weekends vs weekdays
- **Random noise**: Real-world variability

In [ ]:
def generate_retail_sales_data(start_date='2022-01-01', end_date='2024-12-31', num_products=5):
    """
    Generate synthetic retail sales data with realistic patterns.
    
    Parameters:
    -----------
    start_date : str
        Start date for data generation
    end_date : str
        End date for data generation
    num_products : int
        Number of different products to generate data for
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with synthetic sales data
    """
    
    # Create date range
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    
    # Product categories and base demand levels
    products = [
        {'id': 'PROD_001', 'name': 'Laptop', 'category': 'Electronics', 'base_demand': 50},
        {'id': 'PROD_002', 'name': 'Coffee Maker', 'category': 'Appliances', 'base_demand': 80},
        {'id': 'PROD_003', 'name': 'Running Shoes', 'category': 'Sportswear', 'base_demand': 120},
        {'id': 'PROD_004', 'name': 'Winter Jacket', 'category': 'Clothing', 'base_demand': 60},
        {'id': 'PROD_005', 'name': 'Smartphone', 'category': 'Electronics', 'base_demand': 100}
    ][:num_products]
    
    data = []
    
    for product in products:
        for date in date_range:
            # Extract time features
            day_of_week = date.dayofweek  # 0=Monday, 6=Sunday
            month = date.month
            day_of_month = date.day
            quarter = (month - 1) // 3 + 1
            
            # Base demand for this product
            base = product['base_demand']
            
            # Trend component (slight upward trend over time)
            days_since_start = (date - pd.to_datetime(start_date)).days
            trend = days_since_start * 0.02  # Gradual growth
            
            # Seasonal component (varies by product category)
            if product['category'] == 'Electronics':
                # Higher demand in Q4 (holiday season)
                seasonal = 30 * np.sin(2 * np.pi * (month - 1) / 12 + np.pi/2)
            elif product['category'] == 'Clothing':
                # Winter jacket: higher in winter months
                seasonal = -25 * np.sin(2 * np.pi * (month - 1) / 12)
            elif product['category'] == 'Sportswear':
                # Higher in spring/summer
                seasonal = 20 * np.sin(2 * np.pi * (month - 1) / 12)
            else:
                seasonal = 15 * np.sin(2 * np.pi * (month - 1) / 12)
            
            # Day-of-week effect (weekends typically higher)
            if day_of_week in [5, 6]:  # Saturday, Sunday
                dow_effect = 15
            elif day_of_week == 4:  # Friday
                dow_effect = 10
            else:
                dow_effect = 0
            
            # Promotion effect (random promotions)
            is_promotion = np.random.choice([0, 1], p=[0.9, 0.1])  # 10% chance of promotion
            promotion_effect = 40 if is_promotion else 0
            
            # Holiday effect (major retail holidays)
            is_holiday = 0
            # Black Friday (last Friday of November)
            if month == 11 and day_of_week == 4 and day_of_month >= 22:
                is_holiday = 1
                holiday_effect = 100
            # Christmas season
            elif month == 12 and day_of_month >= 15:
                is_holiday = 1
                holiday_effect = 60
            # New Year sales
            elif month == 1 and day_of_month <= 7:
                is_holiday = 1
                holiday_effect = 50
            else:
                holiday_effect = 0
            
            # Random noise (realistic variability)
            noise = np.random.normal(0, 10)
            
            # Calculate total demand (ensure non-negative)
            demand = max(0, base + trend + seasonal + dow_effect + promotion_effect + holiday_effect + noise)
            demand = int(round(demand))
            
            # Price (with occasional discounts during promotions)
            base_prices = {'PROD_001': 899, 'PROD_002': 79, 'PROD_003': 89, 
                          'PROD_004': 149, 'PROD_005': 699}
            base_price = base_prices[product['id']]
            price = base_price * (0.8 if is_promotion else 1.0)
            
            # Inventory level (for context - starts high, depletes with sales)
            # This is simplified - in reality it would be replenished
            
            data.append({
                'date': date,
                'product_id': product['id'],
                'product_name': product['name'],
                'category': product['category'],
                'demand': demand,
                'price': round(price, 2),
                'is_promotion': is_promotion,
                'is_holiday': is_holiday
            })
    
    df = pd.DataFrame(data)
    return df

# Generate the dataset
print("Generating retail sales data...")
df = generate_retail_sales_data(start_date='2022-01-01', end_date='2024-12-31', num_products=5)
print(f"✓ Generated {len(df):,} records for {df['product_id'].nunique()} products")
print(f"✓ Date range: {df['date'].min()} to {df['date'].max()}")

### 3.1 Initial Data Exploration

In [ ]:
# Display basic information
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("First few records:")
print("="*50)
display(df.head(10))

print("\n" + "="*50)
print("Dataset Info:")
print("="*50)
df.info()

print("\n" + "="*50)
print("Statistical Summary:")
print("="*50)
display(df.describe())

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

### 3.2 Exploratory Data Analysis (EDA)

In [ ]:
# Visualize demand over time for each product
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Product Demand Over Time', fontsize=16, fontweight='bold')

products = df['product_id'].unique()

for idx, product in enumerate(products):
    row = idx // 2
    col = idx % 2
    
    product_data = df[df['product_id'] == product].copy()
    product_data = product_data.sort_values('date')
    
    axes[row, col].plot(product_data['date'], product_data['demand'], 
                        linewidth=0.8, alpha=0.7)
    axes[row, col].set_title(f"{product_data['product_name'].iloc[0]} - {product_data['category'].iloc[0]}",
                             fontweight='bold')
    axes[row, col].set_xlabel('Date')
    axes[row, col].set_ylabel('Daily Demand')
    axes[row, col].grid(True, alpha=0.3)
    axes[row, col].tick_params(axis='x', rotation=45)

# Remove empty subplot
fig.delaxes(axes[2, 1])

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- Clear seasonal patterns visible in the data")
print("- Spikes during promotional and holiday periods")
print("- Different products show different demand patterns")

In [ ]:
# Aggregate demand by product
product_summary = df.groupby(['product_id', 'product_name', 'category']).agg({
    'demand': ['sum', 'mean', 'std', 'min', 'max']
}).round(2)

product_summary.columns = ['Total_Demand', 'Avg_Daily_Demand', 'Std_Dev', 'Min_Demand', 'Max_Demand']
product_summary = product_summary.reset_index()

print("\nProduct Demand Summary:")
print("="*80)
display(product_summary)

# Visualize average demand by product
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of average demand
ax1.bar(product_summary['product_name'], product_summary['Avg_Daily_Demand'], 
        color='steelblue', edgecolor='black')
ax1.set_title('Average Daily Demand by Product', fontweight='bold', fontsize=12)
ax1.set_xlabel('Product')
ax1.set_ylabel('Average Daily Demand')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Pie chart of total demand share
ax2.pie(product_summary['Total_Demand'], labels=product_summary['product_name'], 
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Market Share by Total Demand', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze promotion and holiday impact
promotion_impact = df.groupby('is_promotion')['demand'].agg(['mean', 'std', 'count'])
promotion_impact.index = ['Normal Days', 'Promotion Days']

holiday_impact = df.groupby('is_holiday')['demand'].agg(['mean', 'std', 'count'])
holiday_impact.index = ['Normal Days', 'Holiday Days']

print("\nPromotion Impact on Demand:")
print("="*60)
display(promotion_impact)

print("\nHoliday Impact on Demand:")
print("="*60)
display(holiday_impact)

# Calculate lift
promotion_lift = ((promotion_impact.loc['Promotion Days', 'mean'] / 
                   promotion_impact.loc['Normal Days', 'mean']) - 1) * 100

holiday_lift = ((holiday_impact.loc['Holiday Days', 'mean'] / 
                 holiday_impact.loc['Normal Days', 'mean']) - 1) * 100

print(f"\n📊 Promotion Lift: +{promotion_lift:.1f}%")
print(f"📊 Holiday Lift: +{holiday_lift:.1f}%")

In [ ]:
# Day of week analysis
df['day_of_week'] = df['date'].dt.dayofweek
df['day_name'] = df['date'].dt.day_name()

dow_analysis = df.groupby('day_name')['demand'].agg(['mean', 'std']).round(2)

# Reorder to start with Monday
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_analysis = dow_analysis.reindex(day_order)

print("\nDemand by Day of Week:")
print("="*50)
display(dow_analysis)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(dow_analysis.index, dow_analysis['mean'], color='coral', edgecolor='black')
ax.errorbar(dow_analysis.index, dow_analysis['mean'], yerr=dow_analysis['std'], 
            fmt='none', color='black', capsize=5)
ax.set_title('Average Demand by Day of Week (with Std Dev)', fontweight='bold', fontsize=14)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Demand')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Feature Engineering <a name="feature-engineering"></a>

Feature engineering is crucial for time series forecasting. We'll create features that capture:
- **Temporal patterns**: day, month, quarter, year
- **Lag features**: Previous demand values
- **Rolling statistics**: Moving averages, trends
- **Categorical encoding**: Product categories, promotions

In [ ]:
def create_time_features(df):
    """
    Create time-based features from date column.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'date' column
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with added time features
    """
    df = df.copy()
    
    # Basic time features
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    
    # Binary features
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['date'].dt.is_quarter_start.astype(int)
    df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(int)
    
    # Cyclical encoding for month and day_of_week (captures circular nature)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    return df

print("Creating time-based features...")
df = create_time_features(df)
print(f"✓ Added {len([col for col in df.columns if col not in ['date', 'product_id', 'product_name', 'category', 'demand', 'price', 'is_promotion', 'is_holiday']])} time features")

In [ ]:
def create_lag_features(df, product_col='product_id', date_col='date', target_col='demand', lags=[1, 7, 14, 30]):
    """
    Create lag features for time series forecasting.
    
    Lag features capture historical values, which are crucial for prediction.
    For example, lag_1 is yesterday's demand, lag_7 is demand from a week ago.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input DataFrame
    product_col : str
        Column name for product identifier
    date_col : str
        Column name for date
    target_col : str
        Column name for target variable (demand)
    lags : list
        List of lag periods to create
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with lag features
    """
    df = df.copy()
    df = df.sort_values([product_col, date_col])
    
    for lag in lags:
        # Create lag feature for each product separately
        df[f'lag_{lag}'] = df.groupby(product_col)[target_col].shift(lag)
    
    return df

print("Creating lag features...")
df = create_lag_features(df, lags=[1, 7, 14, 30])
print("✓ Added lag features: lag_1, lag_7, lag_14, lag_30")

In [ ]:
def create_rolling_features(df, product_col='product_id', date_col='date', 
                           target_col='demand', windows=[7, 14, 30]):
    """
    Create rolling window statistics (moving averages, std, etc.).
    
    These features help capture trends and variability in demand.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input DataFrame
    product_col : str
        Column name for product identifier
    date_col : str
        Column name for date
    target_col : str
        Column name for target variable
    windows : list
        List of window sizes for rolling calculations
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with rolling features
    """
    df = df.copy()
    df = df.sort_values([product_col, date_col])
    
    for window in windows:
        # Rolling mean (moving average)
        df[f'rolling_mean_{window}'] = df.groupby(product_col)[target_col].transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=1).mean()
        )
        
        # Rolling standard deviation (variability)
        df[f'rolling_std_{window}'] = df.groupby(product_col)[target_col].transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=1).std()
        )
        
        # Rolling min and max
        df[f'rolling_min_{window}'] = df.groupby(product_col)[target_col].transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=1).min()
        )
        
        df[f'rolling_max_{window}'] = df.groupby(product_col)[target_col].transform(
            lambda x: x.shift(1).rolling(window=window, min_periods=1).max()
        )
    
    return df

print("Creating rolling window features...")
df = create_rolling_features(df, windows=[7, 14, 30])
print("✓ Added rolling features for windows: 7, 14, 30 days")

In [ ]:
# Encode categorical variables
print("Encoding categorical variables...")

# One-hot encoding for category
category_dummies = pd.get_dummies(df['category'], prefix='category')
df = pd.concat([df, category_dummies], axis=1)

print(f"✓ Created one-hot encoding for {df['category'].nunique()} categories")

# Display current features
print(f"\nTotal features created: {df.shape[1]}")
print("\nFeature columns:")
print(df.columns.tolist())

In [ ]:
# Handle missing values created by lag and rolling features
print("\nMissing values before handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with NaN values (from initial lag periods)
# In production, you might want to handle these differently
df_clean = df.dropna()

print(f"\nRows before: {len(df):,}")
print(f"Rows after: {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean):,}")

# Update df
df = df_clean.copy()

print("\n✓ Missing values handled")

### 4.1 Feature Importance Preview

Before building models, let's examine correlations to understand which features might be important.

In [ ]:
# Select numerical features for correlation analysis
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove target variable and identifiers
features_for_corr = [col for col in numeric_features if col not in ['demand', 'year']]

# Calculate correlation with target
correlations = df[features_for_corr + ['demand']].corr()['demand'].abs().sort_values(ascending=False)

print("Top 15 Features Correlated with Demand:")
print("="*60)
print(correlations.head(16)[1:])  # Exclude 'demand' itself

# Visualize top correlations
fig, ax = plt.subplots(figsize=(10, 8))
top_corr = correlations.head(16)[1:]
ax.barh(range(len(top_corr)), top_corr.values, color='teal', edgecolor='black')
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index)
ax.set_xlabel('Absolute Correlation with Demand')
ax.set_title('Top 15 Features by Correlation with Demand', fontweight='bold', fontsize=14)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Model Building <a name="model-building"></a>

We'll build and compare multiple models:
1. **Baseline Model**: Simple moving average
2. **Random Forest**: Tree-based ensemble method
3. **XGBoost**: Gradient boosting (often best for tabular data)

### 5.1 Train-Test Split

For time series, we use **temporal split** (not random split) to prevent data leakage.

In [ ]:
# Define features and target
# Exclude non-feature columns
exclude_cols = ['date', 'product_id', 'product_name', 'category', 'demand', 'day_name']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Number of features: {len(feature_cols)}")
print("\nFeatures used for modeling:")
print(feature_cols)

# Prepare features and target
X = df[feature_cols]
y = df['demand']

# Temporal split: Use 80% for training, 20% for testing
# Sort by date to ensure temporal ordering
df_sorted = df.sort_values('date')
split_idx = int(len(df_sorted) * 0.8)

train_df = df_sorted.iloc[:split_idx]
test_df = df_sorted.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df['demand']
X_test = test_df[feature_cols]
y_test = test_df['demand']

print(f"\nTraining set: {len(X_train):,} samples ({train_df['date'].min()} to {train_df['date'].max()})")
print(f"Test set: {len(X_test):,} samples ({test_df['date'].min()} to {test_df['date'].max()})")
print(f"\nTrain/Test split ratio: {len(X_train)/len(X):.1%} / {len(X_test)/len(X):.1%}")

### 5.2 Baseline Model: Moving Average

A simple baseline to compare against ML models.

In [ ]:
# Baseline: Use 7-day moving average as prediction
baseline_predictions = test_df['rolling_mean_7'].values

# Calculate metrics
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline Model (7-Day Moving Average) Performance:")
print("="*60)
print(f"MAE (Mean Absolute Error): {baseline_mae:.2f} units")
print(f"RMSE (Root Mean Squared Error): {baseline_rmse:.2f} units")
print(f"R² Score: {baseline_r2:.4f}")
print("\nInterpretation:")
print(f"- On average, predictions are off by {baseline_mae:.2f} units")
print(f"- The model explains {baseline_r2*100:.2f}% of variance in demand")

### 5.3 Random Forest Model

In [ ]:
print("Training Random Forest model...")
print("This may take a minute...\n")

# Initialize Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,        # Number of trees
    max_depth=15,            # Maximum depth of trees
    min_samples_split=10,    # Minimum samples to split a node
    min_samples_leaf=5,      # Minimum samples in a leaf
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# Calculate metrics
rf_train_mae = mean_absolute_error(y_train, rf_train_pred)
rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rf_train_r2 = r2_score(y_train, rf_train_pred)

rf_test_mae = mean_absolute_error(y_test, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))
rf_test_r2 = r2_score(y_test, rf_test_pred)

print("✓ Random Forest Model Trained\n")
print("Training Set Performance:")
print("="*60)
print(f"MAE: {rf_train_mae:.2f} | RMSE: {rf_train_rmse:.2f} | R²: {rf_train_r2:.4f}")

print("\nTest Set Performance:")
print("="*60)
print(f"MAE: {rf_test_mae:.2f} | RMSE: {rf_test_rmse:.2f} | R²: {rf_test_r2:.4f}")

print(f"\nImprovement over Baseline:")
print(f"MAE: {((baseline_mae - rf_test_mae) / baseline_mae * 100):.1f}% better")
print(f"R²: {((rf_test_r2 - baseline_r2) / abs(baseline_r2) * 100):.1f}% better")

### 5.4 XGBoost Model

XGBoost often performs best for structured/tabular data.

In [ ]:
print("Training XGBoost model...")
print("This may take a minute...\n")

# Initialize XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=200,           # Number of boosting rounds
    max_depth=6,                # Maximum tree depth
    learning_rate=0.1,          # Step size shrinkage
    subsample=0.8,              # Subsample ratio of training instances
    colsample_bytree=0.8,       # Subsample ratio of features
    random_state=42,
    n_jobs=-1
)

# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

# Calculate metrics
xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)
xgb_train_rmse = np.sqrt(mean_squared_error(y_train, xgb_train_pred))
xgb_train_r2 = r2_score(y_train, xgb_train_pred)

xgb_test_mae = mean_absolute_error(y_test, xgb_test_pred)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, xgb_test_pred))
xgb_test_r2 = r2_score(y_test, xgb_test_pred)

print("✓ XGBoost Model Trained\n")
print("Training Set Performance:")
print("="*60)
print(f"MAE: {xgb_train_mae:.2f} | RMSE: {xgb_train_rmse:.2f} | R²: {xgb_train_r2:.4f}")

print("\nTest Set Performance:")
print("="*60)
print(f"MAE: {xgb_test_mae:.2f} | RMSE: {xgb_test_rmse:.2f} | R²: {xgb_test_r2:.4f}")

print(f"\nImprovement over Baseline:")
print(f"MAE: {((baseline_mae - xgb_test_mae) / baseline_mae * 100):.1f}% better")
print(f"R²: {((xgb_test_r2 - baseline_r2) / abs(baseline_r2) * 100):.1f}% better")

## 6. Model Evaluation <a name="model-evaluation"></a>

### 6.1 Model Comparison

In [ ]:
# Create comparison DataFrame
results = pd.DataFrame({
    'Model': ['Baseline (7-day MA)', 'Random Forest', 'XGBoost'],
    'MAE': [baseline_mae, rf_test_mae, xgb_test_mae],
    'RMSE': [baseline_rmse, rf_test_rmse, xgb_test_rmse],
    'R² Score': [baseline_r2, rf_test_r2, xgb_test_r2]
})

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
display(results.round(4))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['MAE', 'RMSE', 'R² Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, metric in enumerate(metrics):
    axes[idx].bar(results['Model'], results[metric], color=colors, edgecolor='black', alpha=0.8)
    axes[idx].set_title(f'{metric} Comparison', fontweight='bold', fontsize=12)
    axes[idx].set_ylabel(metric)
    axes[idx].tick_params(axis='x', rotation=15)
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, v in enumerate(results[metric]):
        axes[idx].text(i, v, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Determine best model
best_model_idx = results['MAE'].idxmin()
best_model = results.loc[best_model_idx, 'Model']
print(f"\n🏆 Best Model: {best_model} (Lowest MAE)")

### 6.2 Prediction vs Actual Visualization

In [ ]:
# Visualize predictions vs actual for XGBoost (best model)
# Select a single product for clearer visualization
sample_product = test_df['product_id'].unique()[0]
product_test = test_df[test_df['product_id'] == sample_product].copy()
product_test = product_test.sort_values('date')

# Get predictions for this product
product_indices = product_test.index
test_indices = test_df.index.tolist()
product_pred_indices = [test_indices.index(idx) for idx in product_indices]

product_xgb_pred = xgb_test_pred[product_pred_indices]

# Plot
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(product_test['date'], product_test['demand'], 
        label='Actual Demand', linewidth=2, marker='o', markersize=3, alpha=0.7)
ax.plot(product_test['date'], product_xgb_pred, 
        label='XGBoost Predictions', linewidth=2, marker='s', markersize=3, alpha=0.7)

ax.set_title(f'Demand Forecast: {product_test["product_name"].iloc[0]} (Test Period)', 
             fontweight='bold', fontsize=14)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Demand (Units)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 6.3 Residual Analysis

In [ ]:
# Calculate residuals (prediction errors)
xgb_residuals = y_test - xgb_test_pred

# Visualize residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual distribution
axes[0].hist(xgb_residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].set_title('Distribution of Prediction Errors', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Residuals vs Predictions
axes[1].scatter(xgb_test_pred, xgb_residuals, alpha=0.5, s=20, color='coral')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Predictions', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted Demand')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical summary of residuals
print("\nResidual Analysis:")
print("="*60)
print(f"Mean Residual: {xgb_residuals.mean():.2f}")
print(f"Std Dev of Residuals: {xgb_residuals.std():.2f}")
print(f"\nResiduals within ±10 units: {(abs(xgb_residuals) <= 10).sum() / len(xgb_residuals) * 100:.1f}%")
print(f"Residuals within ±20 units: {(abs(xgb_residuals) <= 20).sum() / len(xgb_residuals) * 100:.1f}%")

### 6.4 Feature Importance Analysis

In [ ]:
# Get feature importance from XGBoost
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Display top 20 features
print("\nTop 20 Most Important Features:")
print("="*60)
display(feature_importance.head(20))

# Visualize top 15 features
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['Importance'].values, 
        color='forestgreen', edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Top 15 Features by Importance (XGBoost)', fontweight='bold', fontsize=14)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("- Lag features (historical demand) are typically most important")
print("- Rolling statistics capture trends and patterns")
print("- Promotional and holiday flags have significant impact")

## 7. Business Insights & Recommendations <a name="insights"></a>

In [ ]:
# Calculate business metrics
# Assuming average holding cost per unit per day and stockout cost
holding_cost_per_unit_per_day = 0.50  # dollars
stockout_cost_per_unit = 10.0  # lost profit per stockout unit

# Calculate potential cost savings
# Baseline vs XGBoost error reduction
error_reduction = baseline_mae - xgb_test_mae
days_in_test = len(y_test)
products_count = test_df['product_id'].nunique()

# Estimate daily cost savings (simplified calculation)
daily_savings_per_product = error_reduction * (holding_cost_per_unit_per_day + stockout_cost_per_unit) / 2
annual_savings = daily_savings_per_product * 365 * products_count

print("\n" + "="*80)
print("BUSINESS IMPACT ANALYSIS")
print("="*80)

print(f"\n📊 Forecast Accuracy Improvement:")
print(f"   • Prediction error reduced by {error_reduction:.2f} units per day")
print(f"   • {((baseline_mae - xgb_test_mae) / baseline_mae * 100):.1f}% improvement over baseline")

print(f"\n💰 Estimated Annual Cost Savings:")
print(f"   • Per product: ${daily_savings_per_product * 365:,.2f}")
print(f"   • Total ({products_count} products): ${annual_savings:,.2f}")

print(f"\n📈 Key Recommendations:")
print(f"   1. Deploy XGBoost model for daily demand forecasting")
print(f"   2. Update forecasts daily using latest demand data")
print(f"   3. Adjust safety stock based on prediction intervals")
print(f"   4. Monitor model performance weekly and retrain monthly")
print(f"   5. Integrate forecasts with inventory management system")

print(f"\n🎯 Business Benefits:")
print(f"   • Reduced inventory holding costs")
print(f"   • Fewer stockouts and lost sales")
print(f"   • Improved customer satisfaction")
print(f"   • Better cash flow management")
print(f"   • Data-driven inventory decisions")

### 7.1 Product-Level Insights

In [ ]:
# Analyze performance by product
test_df_with_pred = test_df.copy()
test_df_with_pred['prediction'] = xgb_test_pred
test_df_with_pred['error'] = abs(test_df_with_pred['demand'] - test_df_with_pred['prediction'])

product_performance = test_df_with_pred.groupby(['product_id', 'product_name']).agg({
    'demand': 'mean',
    'prediction': 'mean',
    'error': 'mean'
}).round(2)

product_performance.columns = ['Avg_Actual_Demand', 'Avg_Predicted_Demand', 'Avg_Absolute_Error']
product_performance['Error_Percentage'] = (
    product_performance['Avg_Absolute_Error'] / product_performance['Avg_Actual_Demand'] * 100
).round(2)

product_performance = product_performance.reset_index()

print("\nForecast Accuracy by Product:")
print("="*80)
display(product_performance)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(product_performance))
width = 0.35

ax.bar(x - width/2, product_performance['Avg_Actual_Demand'], width, 
       label='Actual', color='steelblue', edgecolor='black', alpha=0.8)
ax.bar(x + width/2, product_performance['Avg_Predicted_Demand'], width, 
       label='Predicted', color='coral', edgecolor='black', alpha=0.8)

ax.set_xlabel('Product', fontsize=12)
ax.set_ylabel('Average Demand', fontsize=12)
ax.set_title('Actual vs Predicted Average Demand by Product', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(product_performance['product_name'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Deployment Considerations <a name="deployment"></a>

### Key Steps for Production Deployment:

1. **Model Serialization**
   - Save trained model using joblib or pickle
   - Version control models

2. **API Development**
   - Create REST API (e.g., Flask, FastAPI)
   - Handle input validation
   - Return predictions with confidence intervals

3. **Monitoring & Retraining**
   - Track prediction accuracy over time
   - Detect model drift
   - Automate periodic retraining

4. **Integration**
   - Connect to inventory management system
   - Automate daily forecast updates
   - Create dashboards for stakeholders

5. **Scalability**
   - Handle multiple products efficiently
   - Use cloud services (AWS SageMaker, Azure ML)
   - Implement caching for frequently requested forecasts

In [ ]:
# Example: Save the trained model
import joblib

# Save model
model_filename = 'demand_forecasting_xgboost_model.pkl'
joblib.dump(xgb_model, model_filename)
print(f"✓ Model saved as '{model_filename}'")

# Example: Load model and make prediction
loaded_model = joblib.load(model_filename)

# Make a sample prediction
sample_input = X_test.iloc[0:1]
sample_prediction = loaded_model.predict(sample_input)

print(f"\nSample Prediction:")
print(f"Predicted Demand: {sample_prediction[0]:.2f} units")
print(f"Actual Demand: {y_test.iloc[0]:.2f} units")
print(f"Error: {abs(sample_prediction[0] - y_test.iloc[0]):.2f} units")

## 9. Conclusion

### What We Accomplished:

1. ✅ **Business Understanding**: Defined the inventory optimization problem
2. ✅ **Data Generation**: Created realistic synthetic retail sales data
3. ✅ **Exploratory Analysis**: Understood demand patterns and seasonality
4. ✅ **Feature Engineering**: Created time-based, lag, and rolling features
5. ✅ **Model Development**: Built and compared multiple ML models
6. ✅ **Evaluation**: Assessed model performance using multiple metrics
7. ✅ **Business Impact**: Quantified potential cost savings

### Key Learnings:

- **Lag features** (historical demand) are crucial for time series forecasting
- **Rolling statistics** help capture trends and patterns
- **XGBoost** outperformed baseline and Random Forest models
- **Proper validation** using temporal splits prevents data leakage
- **Feature engineering** is often more important than model selection

### Next Steps:

1. Experiment with deep learning models (LSTM, Transformer)
2. Incorporate external data (weather, economic indicators)
3. Implement prediction intervals for uncertainty quantification
4. Build automated retraining pipelines
5. Create interactive dashboards for business stakeholders

---

### Additional Resources:

- **Scikit-learn Documentation**: https://scikit-learn.org/
- **XGBoost Documentation**: https://xgboost.readthedocs.io/
- **Time Series Forecasting**: "Forecasting: Principles and Practice" by Hyndman & Athanasopoulos
- **Retail Analytics**: McKinsey on Retail Analytics
- **MLOps**: "Building Machine Learning Powered Applications" by Emmanuel Ameisen

---

## 📚 Learning Exercises

Try these exercises to deepen your understanding:

1. **Modify seasonality patterns** for different product categories
2. **Add new features** like competitor pricing or weather data
3. **Experiment with hyperparameters** to improve model performance
4. **Build a Prophet model** and compare with XGBoost
5. **Create prediction intervals** to quantify uncertainty
6. **Implement cross-validation** using TimeSeriesSplit
7. **Build a simple Flask API** to serve predictions
8. **Create an interactive dashboard** using Plotly or Streamlit

---